# fetch all stock

In [12]:
import os,json
import pandas as pd
import requests
import psycopg2
from psycopg2.extras import execute_values,json
from dotenv import load_dotenv

In [4]:
load_dotenv()

PG_HOST = os.getenv("PG_HOST")
PG_PORT = int(os.getenv("PG_PORT"))
PG_DB   = os.getenv("PG_DB")
PG_USER = os.getenv("PG_USER")
PG_PWD  = os.getenv("PG_PASSWORD")
token = os.getenv("finmind_token")

In [6]:
url = "https://api.finmindtrade.com/api/v4/data"
# token = fin_api_token.finmind_token
headers = {"Authorization": f"Bearer {token}"}
parameter = {
    "dataset": "TaiwanStockInfo",
}
resp = requests.get(url, headers=headers, params=parameter, timeout=60)
resp.raise_for_status()
js = resp.json()
assert js.get("status") == 200, js.get("msg") #check if its wrong

df_info = pd.DataFrame(js["data"])
print("rows:", len(df_info), df_info.columns.tolist())

#normalize
def normalize(df: pd.DataFrame)-> pd.DataFrame:
    out = pd.DataFrame(
        {
            "stock_id": df_info["stock_id"].astype(str).str.strip(),
            "stock_name": df_info["stock_name"],
            "industry_category": df_info["industry_category"],
            "list_type": df_info["type"],
            "list_date": pd.to_datetime(df_info["date"], errors="coerce").dt.date
        }
    )
    out["raw"] = df_info.apply(lambda r:json.loads(r.to_json(force_ascii=False)),axis=1)
    out = out[out["stock_id"].notna() & (out["stock_id"].str.len() > 0)].copy()
    
    return out

df_info_norms = normalize(df_info)
df_info_norms.head()

rows: 3975 ['industry_category', 'stock_id', 'stock_name', 'type', 'date']


,stock_id,stock_name,industry_category,list_type,list_date,raw
0,3687,歐買尬,文化創意業,tpex,2020-06-03,"{'industry_category': '文化創意業', 'stock_id': '36..."
1,5481,新華,電子零組件業,tpex,2020-06-03,"{'industry_category': '電子零組件業', 'stock_id': '5..."
2,3629,地心引力,光電業,tpex,2020-06-03,"{'industry_category': '光電業', 'stock_id': '3629..."
3,5450,寶聯通,電腦及週邊設備業,tpex,2020-06-03,"{'industry_category': '電腦及週邊設備業', 'stock_id': ..."
4,6238,勝麗,其他電子類,tpex,2020-06-13,"{'industry_category': '其他電子類', 'stock_id': '62..."


In [9]:
df_info_norms.tail(20)

,stock_id,stock_name,industry_category,list_type,list_date,raw
3955,Tourism,觀光事業類指數,Index,twse,NaT,"{'industry_category': 'Index', 'stock_id': 'To..."
3956,Plastics,塑膠類指數,Index,twse,NaT,"{'industry_category': 'Index', 'stock_id': 'Pl..."
3957,GlassCeramic,玻璃陶瓷類指數,Index,twse,NaT,"{'industry_category': 'Index', 'stock_id': 'Gl..."
3958,BuildingMaterialConstruction,建材營造類指數,Index,twse,NaT,"{'industry_category': 'Index', 'stock_id': 'Bu..."
3959,FinancialInsurance,金融保險類指數,Index,twse,NaT,"{'industry_category': 'Index', 'stock_id': 'Fi..."
3960,ElectronicProductsDistribution,電子通路類指數,Index,twse,NaT,"{'industry_category': 'Index', 'stock_id': 'El..."
3961,ElectronicPartsComponents,電子零組件類指數,Index,twse,NaT,"{'industry_category': 'Index', 'stock_id': 'El..."
3962,Electronic,電子類指數,Index,twse,NaT,"{'industry_category': 'Index', 'stock_id': 'El..."
3963,ElectricMachinery,電機機械類指數,Index,twse,NaT,"{'industry_category': 'Index', 'stock_id': 'El..."
3964,ElectricalCable,電器電纜類指數,Index,twse,NaT,"{'industry_category': 'Index', 'stock_id': 'El..."


In [10]:
df_info_norms.to_csv("all_stock.csv")

In [17]:
dups = df_info_norms[df_info_norms['stock_id'].duplicated(keep=False)] \
          .sort_values('stock_id')
print(len(dups), "rows duplicated")
dups.head(20)

1898 rows duplicated


,stock_id,stock_name,industry_category,list_type,list_date,raw
2767,006201,元大富櫃50,上櫃ETF,tpex,2025-09-30,"{'industry_category': '上櫃ETF', 'stock_id': '00..."
530,006201,元大富櫃50,上櫃指數股票型基金(ETF),tpex,2024-12-30,"{'industry_category': '上櫃指數股票型基金(ETF)', 'stock..."
599,00679B,元大美債20年,上櫃指數股票型基金(ETF),tpex,2024-12-30,"{'industry_category': '上櫃指數股票型基金(ETF)', 'stock..."
2820,00679B,元大美債20年,上櫃ETF,tpex,2025-09-30,"{'industry_category': '上櫃ETF', 'stock_id': '00..."
2828,00687B,國泰20年美債,上櫃ETF,tpex,2025-09-30,"{'industry_category': '上櫃ETF', 'stock_id': '00..."
598,00687B,國泰20年美債,上櫃指數股票型基金(ETF),tpex,2024-12-30,"{'industry_category': '上櫃指數股票型基金(ETF)', 'stock..."
2835,00694B,富邦美債1-3,上櫃ETF,tpex,2025-09-30,"{'industry_category': '上櫃ETF', 'stock_id': '00..."
596,00694B,富邦美債1-3,上櫃指數股票型基金(ETF),tpex,2024-12-30,"{'industry_category': '上櫃指數股票型基金(ETF)', 'stock..."
593,00695B,富邦美債7-10,上櫃指數股票型基金(ETF),tpex,2024-12-30,"{'industry_category': '上櫃指數股票型基金(ETF)', 'stock..."
2836,00695B,富邦美債7-10,上櫃ETF,tpex,2025-09-30,"{'industry_category': '上櫃ETF', 'stock_id': '00..."


In [20]:
# ---------- 切成兩份 ----------
# A) dim_stock：同一 stock_id 只留一筆（優先 twse > tpex > emerging，再比 list_date 較新）
prior = {"twse":3,"tpex":2,"emerging":1}
df_info_norms["prior"] = df_info_norms["list_type"].str.lower().map(prior).fillna(0).astype(int)
dim_stock = (df_info_norms.sort_values(["stock_id","prio","list_date"])
                .drop_duplicates("stock_id", keep="last")
                .drop(columns=["prior","industry_category"])
                .reset_index(drop=True))

# B) dim_stock_category：保留所有 (stock_id, industry_category) 組合（去重）
dim_cat = (df_info_norms[["stock_id","industry_category"]]
              .dropna(subset=["industry_category"])
              .drop_duplicates()
              .reset_index(drop=True))

print(f"dim_stock rows: {len(dim_stock)} ; dim_stock_category rows: {len(dim_cat)}")

# ---------- UPSERT 進資料庫 ----------
def to_none(v): 
    return None if (pd.isna(v)) else v

upsert_dim_stock_sql = """
INSERT INTO stock.dim_stock (stock_id, stock_name, list_type, list_date, raw)
VALUES %s
ON CONFLICT (stock_id) DO UPDATE SET
  stock_name = EXCLUDED.stock_name,
  list_type  = EXCLUDED.list_type,
  list_date  = EXCLUDED.list_date,
  raw        = EXCLUDED.raw;
"""

upsert_dim_cat_sql = """
INSERT INTO stock.dim_stock_category (stock_id, industry_category)
VALUES %s
ON CONFLICT (stock_id, industry_category) DO NOTHING;
"""

with psycopg2.connect(host=PG_HOST, port=PG_PORT, dbname=PG_DB, user=PG_USER, password=PG_PWD) as conn:
    with conn.cursor() as cur:
        # A) dim_stock
        values_stock = [
            (str(r.stock_id), r.stock_name, r.list_type, to_none(r.list_date), Json(r.raw))
            for r in dim_stock[["stock_id","stock_name","list_type","list_date","raw"]].itertuples(index=False)
        ]
        execute_values(cur, upsert_dim_stock_sql, values_stock, page_size=5000)

        # B) dim_stock_category
        values_cat = [ (str(r.stock_id), r.industry_category)
                       for r in dim_cat.itertuples(index=False) ]
        execute_values(cur, upsert_dim_cat_sql, values_cat, page_size=10000)

    conn.commit()

print(f"寫入完成  dim_stock={len(values_stock)}；dim_stock_category={len(values_cat)}")

KeyError: 'prio'